# 08 — Embeddings & Vector Stores

In notebook 07 we loaded and chunked documents. Now we make those chunks **searchable by
meaning**. The trick is to convert text into numbers — **embeddings** — and store them in a
**vector store** that can find the chunks most similar to a question.

This is the engine under semantic search and RAG. In this notebook:

1. What an embedding is, with Gemini's embedding model.
2. Measuring **semantic similarity** between texts.
3. Building an in-memory **vector store** from document chunks.
4. **Searching** it, with scores.
5. The **retriever** interface — the standard way RAG asks for relevant chunks.
6. A note on persistent, production-grade vector stores.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in your .env file first."
assert os.path.exists("data/lumina_handbook.md"), "Run `python make_data.py` first."
print("Ready.")

## 1. What is an embedding?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of a piece
of text. The model is trained so that texts with similar meaning land at nearby points in
this high-dimensional space. "Where is the warranty info?" and "How long is the guarantee?"
end up close together even though they share few words.

Gemini exposes an embedding model through `GoogleGenerativeAIEmbeddings`. Two methods:

- **`embed_query(text)`** — embed a single string (a user's question).
- **`embed_documents([texts])`** — embed many strings at once (your chunks).

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# text-embedding-004 is a stable Gemini embedding model. If it is ever unavailable,
# try "models/gemini-embedding-001" or "models/embedding-001" — the code is identical.
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

vector = embeddings.embed_query("How long is the Pixel warranty?")
print("A query becomes a vector of length:", len(vector))
print("First 8 numbers:", [round(x, 4) for x in vector[:8]])

So every piece of text becomes a fixed-length vector. The length (the *dimension*) is a
property of the model. We never read these numbers ourselves — we let math compare them.

## 2. Similarity = closeness of vectors

The standard way to compare two embeddings is **cosine similarity**: it ranges from -1 to
1, where 1 means "point in the same direction" (very similar) and 0 means "unrelated".

Let's embed a few sentences and confirm the model groups them by meaning, not by shared
words.

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

sentences = [
    "How long is the warranty on the robot?",   # 0
    "What does the guarantee period cover?",     # 1  (similar meaning to 0)
    "My favorite pizza topping is mushroom.",    # 2  (unrelated)
]
vecs = embeddings.embed_documents(sentences)

print("sim(0,1) warranty vs guarantee :", round(cosine(vecs[0], vecs[1]), 3))
print("sim(0,2) warranty vs pizza     :", round(cosine(vecs[0], vecs[2]), 3))

The warranty/guarantee pair scores much higher than warranty/pizza — even though
"warranty" and "guarantee" are different words. That semantic matching is exactly what
makes retrieval work: a user's paraphrased question still finds the right chunk.

## 3. A vector store does this at scale

Computing cosine similarity by hand against thousands of chunks would be tedious. A
**vector store** stores all your chunk embeddings and finds the nearest ones to a query
efficiently.

We'll use **`InMemoryVectorStore`**, which ships with `langchain-core` and needs no extra
install — perfect for learning. First, rebuild the chunks from notebook 07.

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = DirectoryLoader(
    "data", glob="**/*.md", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
).load()

chunks = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80).split_documents(docs)
print("Prepared", len(chunks), "chunks to index.")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore

# This embeds every chunk (one API call batch) and stores the vectors.
store = InMemoryVectorStore.from_documents(chunks, embedding=embeddings)
print("Indexed", len(chunks), "chunks into the vector store.")

## 4. Searching the store

`similarity_search(query, k=...)` embeds the query and returns the `k` most similar
chunks. Ask a question whose answer lives in the handbook and see what comes back.

In [ ]:
results = store.similarity_search("How long is the warranty?", k=2)

for i, doc in enumerate(results, 1):
    print(f"--- result {i} (source: {os.path.basename(doc.metadata['source'])}) ---")
    print(doc.page_content.strip()[:300], "\n")

The top chunk should be the warranty paragraph mentioning "24-month limited warranty"
— retrieved by meaning, not keyword. Use `similarity_search_with_score` to also see *how*
close each match is.

In [ ]:
scored = store.similarity_search_with_score("Can I get a refund?", k=3)
for doc, score in scored:
    snippet = doc.page_content.strip().replace("\n", " ")[:70]
    print(f"score={score:.3f}  {snippet}...")

> Score conventions vary by store (some report distance where *lower* is better, others
> similarity where *higher* is better). For `InMemoryVectorStore` the score is a cosine
> similarity, so higher means more relevant. Always check the docs for the store you use.

## 5. The retriever interface

In a real chain you don't call `similarity_search` directly. Instead you turn the store
into a **retriever** with `.as_retriever()`. A retriever is a `Runnable`: you `.invoke()`
it with a query string and it returns a list of `Document`s. Because it's a Runnable, it
drops straight into an LCEL chain — which is exactly what we'll do in notebook 09.

In [ ]:
retriever = store.as_retriever(search_kwargs={"k": 2})

found = retriever.invoke("What languages does Pixel Pro support?")
for doc in found:
    print("-", doc.page_content.strip()[:120], "...")

`search_kwargs={"k": 2}` controls how many chunks come back. Retrievers also support
other search types (for example `search_type="mmr"` for *maximal marginal relevance*,
which diversifies results to reduce near-duplicate chunks).

## 6. Persistence and production stores

`InMemoryVectorStore` keeps vectors in RAM — they vanish when the kernel restarts, and it
won't scale to millions of chunks. Real systems use a dedicated vector store that persists
to disk or a server. Common choices:

- **Chroma** (`langchain-chroma`) — easy local persistence.
- **FAISS** (`langchain-community`) — fast in-process similarity search; saves to disk.
- **pgvector** (`langchain-postgres`) — vectors inside PostgreSQL.
- Managed services like Pinecone, Weaviate, Milvus.

The good news: they all expose the **same** `add_documents` / `similarity_search` /
`as_retriever` interface. Swap the store class and the rest of your code is unchanged —
the same swappability you saw with chat models.

## Your turn

Five exercises around semantic search you'd ship — a product search, a source-filtered query,
choosing `k` from scores, diversifying results with MMR, and finding duplicate tickets. Try
each in a blank cell before opening its solution.

**Exercise 1 — Semantic product search.** Build a small `InMemoryVectorStore` from a handful
of product descriptions, then find the best match for a need phrased in totally different
words ("something to keep my coffee warm").

*Sample run (illustrative):*

```text
Insulated steel mug that keeps drinks hot for 6 hours.
```

<details><summary>Show solution</summary>

```python
from langchain_core.vectorstores import InMemoryVectorStore

products = [
    "Insulated steel mug that keeps drinks hot for 6 hours.",
    "Noise-cancelling over-ear headphones with 30-hour battery.",
    "Compact umbrella that fits in a backpack.",
    "Fast wireless charger for phones and earbuds.",
]
shop = InMemoryVectorStore.from_texts(products, embedding=embeddings)

hit = shop.similarity_search("something to keep my coffee warm", k=1)[0]
print(hit.page_content)
```

The query shares no keywords with the mug description, yet it matches — that's semantic search.

</details>

**Exercise 2 — Search only one source.** Query the `store` from section 3 but use a metadata
`filter` so results can only come from the FAQ file.

*Sample run (illustrative):*

```text
lumina_faq.md → To reset Pixel, hold the power button for 10 seconds until the light ...
lumina_faq.md → If Pixel is unresponsive, a factory reset restores it to default ...
```

<details><summary>Show solution</summary>

```python
faq_only = store.similarity_search(
    "How do I reset the robot?", k=2,
    filter=lambda doc: "faq" in os.path.basename(doc.metadata["source"]),
)
for d in faq_only:
    print(os.path.basename(d.metadata["source"]), "→", d.page_content.strip()[:60], "...")
```

Filtering by metadata is how you scope a search — "only the FAQ", "only 2024 docs", and so on.

</details>

**Exercise 3 — Choose `k` from the scores.** Run `similarity_search_with_score` with `k=4`
for a question and look at where the scores drop off — that gap tells you how many chunks are
actually relevant enough to feed a model.

*Sample run (illustrative — scores vary):*

```text
0.811  Returns are accepted within 30 days of delivery for a full refund...
0.642  The 24-month limited warranty covers manufacturing defects...
0.498  Support hours are 9am to 6pm, Monday through Friday...
0.391  Pixel ships in three colors and two storage tiers...
```

<details><summary>Show solution</summary>

```python
for doc, score in store.similarity_search_with_score("What is the return policy?", k=4):
    print(f"{score:.3f}  {doc.page_content.strip()[:60]}...")
```

A sharp fall after the first one or two results means a small `k` is enough; padding it just
adds noise and tokens.

</details>

**Exercise 4 — Diversify results with MMR.** Turn the store into an MMR retriever
(`search_type="mmr"`) so the chunks it returns cover different angles instead of repeating the
same paragraph.

*Sample run (illustrative):*

```text
- Pixel Pro features a 4-hour battery, LiDAR navigation, and voice control...
- Connectivity includes Wi-Fi 6 and Bluetooth 5.2, with an optional hub...
- Pixel supports five languages and ships with a companion mobile app...
```

<details><summary>Show solution</summary>

```python
diverse = store.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 8})
for d in diverse.invoke("Tell me about Pixel's features"):
    print("-", d.page_content.strip()[:70], "...")
```

MMR pulls candidates (`fetch_k`) then picks a varied subset (`k`) — useful when chunks overlap.

</details>

**Exercise 5 — Find duplicate support tickets.** Embed a short list of tickets and use
`cosine` to find the most similar pair — the basis of "this looks like an existing ticket"
detection.

*Sample run (illustrative):*

```text
most similar (sim=0.861):
  • I can't log into my account.
  • Login isn't working for me.
```

<details><summary>Show solution</summary>

```python
tickets = [
    "I can't log into my account.",
    "Login isn't working for me.",
    "How do I change my billing address?",
    "The app crashes on startup.",
]
vt = embeddings.embed_documents(tickets)

best_pair, best_sim = None, -1.0
for i in range(len(tickets)):
    for j in range(i + 1, len(tickets)):
        s = cosine(vt[i], vt[j])
        if s > best_sim:
            best_pair, best_sim = (i, j), s

i, j = best_pair
print(f"most similar (sim={best_sim:.3f}):\n  • {tickets[i]}\n  • {tickets[j]}")
```

The two login tickets score highest — embeddings catch that they mean the same thing.

</details>

## Summary

- An **embedding** maps text to a vector; similar meanings produce nearby vectors.
- `GoogleGenerativeAIEmbeddings` provides `embed_query` and `embed_documents`.
- **Cosine similarity** measures closeness; semantic matches beat keyword overlap.
- A **vector store** indexes chunk embeddings and answers `similarity_search` /
  `similarity_search_with_score`.
- **`.as_retriever()`** exposes the store as a `Runnable` that returns `Document`s — the
  plug RAG chains use.
- Swap `InMemoryVectorStore` for Chroma/FAISS/pgvector in production; the interface stays
  the same.

**Next:** [09 — Retrieval-Augmented Generation](09_Retrieval_Augmented_Generation.ipynb).
We finally connect retrieval to the model and answer questions grounded in our documents.